# Wieże Hanoi: przestrzeń stanów, graf i przeszukiwanie – interaktywnie
**Wprowadzenie do AI · laboratorium 4**

Plan:
0. **Powtórka z Lab 2 i Lab 3** – od definicji przestrzeni stanów do przeszukiwania grafu (BFS i DFS),
1. **Skąd wzięły się Wieże Hanoi?** – historia, legenda o końcu świata i co ta zagadka pomogła udowodnić w matematyce, informatyce oraz kognitywistyce,
2. **Wieże Hanoi jako przestrzeń stanów i graf** – reprezentacja stanu jako krotki, rozmiar przestrzeni $k^n$, trójkąt Sierpińskiego oraz zachowanie BFS i DFS,
3. **🗼 Wyzwanie interaktywne** – rozwiąż Wieże Hanoi samodzielnie dla wybranej liczby krążków ($n \ge 2$) i słupków ($k \ge 2$), a potem porównaj swoją ścieżkę z **BFS (najkrótsza ścieżka)** oraz **DFS**.

> **Jak uruchomić:** uruchom komórki po kolei (`Shift+Enter`). Pierwsza komórka (ukryta) sama pobierze potrzebny kod z GitHuba.

In [ ]:
#@title ⚙️ Przygotowanie – uruchom tę komórkę (kod ukryty)
import os, sys, subprocess
REPO = "https://github.com/BartekDaniluk/kognitywistyka_intro_to_AI"
if "google.colab" in sys.modules:
    root = "/content/kognitywistyka_intro_to_AI"
    if not os.path.exists(root):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, root], check=True)
    else:
        subprocess.run(["git", "-C", root, "pull", "-q"], check=False)
    for sub in ("lab02_stany", "lab03_przeszukiwanie", "lab04_hanoi"):
        p = os.path.join(root, sub)
        if p not in sys.path:
            sys.path.insert(0, p)
else:
    for sub in (".", "../lab02_stany", "../lab03_przeszukiwanie", "lab02_stany", "lab03_przeszukiwanie", "lab04_hanoi"):
        p = os.path.abspath(sub)
        if os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)
%matplotlib inline
import lab_states as ls
import lab_search as lsearch
import lab_hanoi as lh
ls.set_language("pl")
print("Gotowe ✔")

## 0. Powtórka: co już umiemy? (Przestrzenie stanów + Przeszukiwanie grafu)

Zanim zmierzymy się z nowym problemem, przypomnijmy sobie nasze dwa najważniejsze narzędzia z poprzednich zajęć:

### Krok A: Od problemu do grafu przestrzeni stanów (Lab 2)
Każdą łamigłówkę czy zadanie planowania opisujemy formalnie przez **5 składników**:
1. **Stany ($S$)** – zwięzły, jednoznaczny opis sytuacji (zwykle jako krotka zmiennych, z pominięciem nieistotnych szczegółów – *abstrakcja*).
2. **Stan początkowy ($s_0$)** – punkt startu.
3. **Test celu** – warunek mówiący, kiedy zadanie jest rozwiązane.
4. **Akcje i model przejścia** – reguły `(stan | warunek) → nowy stan` mówiące, jakie ruchy są dozwolone i do czego prowadzą.
5. **Koszt akcji ($c$)** – gdy każdy ruch kosztuje `1`, najtańsze rozwiązanie to po prostu **najkrótsza ścieżka** (najmniejsza liczba ruchów).

Z takiej definicji automatycznie powstaje **graf przestrzeni stanów**: stany stają się **węzłami**, a dozwolone ruchy – **krawędziami**.

### Krok B: Jak przeszukujemy ten graf? (Lab 3)
Gdy mamy już graf (lub reguły pozwalające generować sąsiadów), możemy go przeszukać:
* 🌊 **BFS (*Breadth-First Search* – przeszukiwanie wszerz)**:
  * Używa **kolejki FIFO** (*kto pierwszy wejdzie, ten pierwszy wychodzi*).
  * Sprawdza stany **warstwa po warstwie** (najpierw odległe o 1 ruch, potem o 2 ruchy, potem o 3 ruchy…).
  * **Gwarancja:** gdy każda akcja kosztuje tyle samo ($c=1$), BFS **zawsze znajduje najkrótszą możliwą ścieżkę** do celu!
* 🕳️ **DFS (*Depth-First Search* – przeszukiwanie w głąb)**:
  * Używa **stosu LIFO** (jawnego w pętli albo ukrytego na **stosie wywołań rekurencyjnych**).
  * Idzie jak najgłębiej jedną ścieżką, a gdy trafi na ślepy zaułek (lub stan już odwiedzony), **cofa się (*backtracking*)**.
  * **Pułapka:** w grafie z cyklami DFS potrafi „wpaść w wir” i dotrzeć do celu bardzo długą, okrężną drogą!

Dzisiaj zobaczymy oba te mechanizmy w pełnej krasie na jednym z najsłynniejszych problemów w historii sztucznej inteligencji i kognitywistyki: **Wieżach Hanoi**.

## 1. Wieże Hanoi – skąd się wzięły i co pomogły udowodnić?

### Zasady łamigłówki
Mamy **$k$ pionowych słupków** (klasycznie $k=3$: **A**, **B**, **C**) oraz **$n$ krążków** o różnych średnicach ($1$ – najmniejszy, $2$, …, $n$ – największy). Na początku wszystkie krążki leżą na pierwszym słupku (**A**), ułożone od największego na dole do najmniejszego na górze.

Chcemy przenieść całą wieżę na ostatni słupek (docelowy), przestrzegając **tylko dwóch reguł**:
1. W jednym ruchu wolno przenieść **tylko jeden, wierzchołkowy krążek** z jednego słupka na drugi.
2. **Nigdy nie wolno położyć większego krążka na mniejszy!**

---

### 📜 Historia i legenda o końcu świata (1883)
Łamigłówkę wymyślił w **1883 roku** francuski matematyk **Édouard Lucas** (ten sam, który badał ciąg Fibonacciego i wprowadził jego nazwę). Pierwsze drewniane pudełka z zabawką sprzedawano w Paryżu pod tajemniczym pseudonimem *„Prof. N. Claus de Siam, mandaryn kolegium Li-Sou-Stian”* – co było po prostu anagramem od ***Lucas d’Amiens*** oraz nazwy jego liceum ***Saint-Louis***!

Aby zainteresować kupujących, Lucas dołączył do instrukcji wymyśloną przez siebie **legendę o Wielkiej Świątyni w Benares (Waranasi)**:
> *Pod kopułą świątyni bóg Brahma ustawił trzy diamentowe igły, a na jednej z nich umieścił **64 krążki ze szczerego złota**. Kapłani dniem i nocą przekładają po jednym krążku zgodnie z niezmiennymi prawami Brahmy. Gdy wszystkie 64 krążki zostaną przełożone na ostatnią igłę – świątynia obróci się w pył i nastąpi koniec świata.*

Lucas nie wymyślił tej historii wyłącznie dla reklamy. Chciał w obrazowy sposób pokazać ludziom, czym jest **wzrost wykładniczy (eksplozja kombinatoryczna)**:
* Dla 3 krążków wystarczy $2^3 - 1 = 7$ ruchów.
* Dla 64 krążków minimalna liczba ruchów to:
  $$2^{64} - 1 = 18\,446\,744\,073\,709\,551\,615$$
* Gdyby kapłani wykonywali bezbłędnie **1 ruch w każdą sekundę**, przełożenie wieży zajęłoby im około **585 miliardów lat** – czyli ponad **40 razy dłużej niż dotychczasowy wiek całego Wszechświata**!

---

### 🔬 Co Wieże Hanoi pomogły zbadać i udowodnić?

Choć zaczęły jako zabawka, Wieże Hanoi okazały się niezwykle ważnym modelem badawczym w trzech dziedzinach:

#### 1. Matematyka i Informatyka: rekurencja, fraktale i zagadka 4 słupków (rozwiązana dopiero w 2014 r.!)
* **Modelowy dowód indukcyjny i rekurencja:** Lucas pokazał, że problem ma idealną strukturę samopodobną. Aby przenieść $n$ krążków ze słupka A na C (przy użyciu B), *musisz* najpierw odsłonić największy krążek $n$, czyli przenieść $n-1$ mniejszych krążków na słupek B, potem przełożyć krążek $n$ z A na C (1 ruch), a na końcu przenieść $n-1$ krążków z B na C. Daje to równanie rekurencyjne:
  $$L(n) = 2 \cdot L(n-1) + 1 \quad \Longrightarrow \quad L(n) = 2^n - 1$$
* **Trójkąt Sierpińskiego (fraktale):** Gdy narysujemy graf wszystkich stanów Wież Hanoi dla 3 słupków, okazuje się on idealnym **trójkątem Sierpińskiego** (zobaczysz to za chwilę na wykresie!).
* **Zagadka 4 i więcej słupków (*Hipoteza Frame’a–Stewarta*, 1941–2014):** Co się stanie, gdy dodamy czwarty słupek ($k=4$)? Zagadka robi się znacznie łatwiejsza (mniej ruchów!), ale przez **73 lata** nikt na świecie nie potrafił matematycznie udowodnić, jaka jest minimalna liczba ruchów! Dopiero w **2014 roku** francuski matematyk **Thierry Bousch** udowodnił formalnie, że dla $k=4$ słupków algorytm zaproponowany w 1941 r. przez Frame’a i Stewarta faktycznie daje najkrótszą ścieżkę. Co ciekawe, dla **$k \ge 5$ słupków** ogólny dowód matematyczny wciąż jest **problemem otwartym** – natomiast przeszukiwanie **BFS** potrafi znaleźć najkrótszą ścieżkę dla zadanych $n$ i $k$ automatycznie!

#### 2. Kognitywistyka i Sztuczna Inteligencja: jak człowiek rozwiązuje problemy?
* **Newell i Simon (*Human Problem Solving*, 1972):** Ojcowie AI i kognitywistyki – Allen Newell i Herbert Simon (laureat Nagrody Nobla i Nagrody Turinga) – użyli Wież Hanoi jako głównego poligonu do badania ludzkiego myślenia. Zaobserwowali, że człowiek nie przeszukuje w głowie całego grafu wszerz jak BFS (nasza pamięć robocza ma zbyt małą pojemność!). Zamiast tego umysł stosuje **analizę środków i celów (*Means-Ends Analysis*)** oraz buduje **hierarchię podcelów (*subgoaling*)**:
  > *„Chcę przenieść krążek 3 na C → przeszkadzają mi krążki 1 i 2 → tworzę podcel: przenieś wieżę {1, 2} na słupek pomocniczy B…”*
* **Diagnostyka neuropsychologiczna (płaty czołowe):** W psychologii klinicznej Wieże Hanoi (oraz ich wariant – *Wieża Londyńska*, Shallice 1982) są standardowym testem **funkcji wykonawczych kory przedczołowej**. Aby rozwiązać łamigłówkę, badany musi **powstrzymać impuls** (*hamowanie poznawcze / inhibition*): żeby przełożyć duży krążek na słupek docelowy, trzeba najpierw przenieść mniejsze krążki na słupek *inny* niż docelowy (czyli pozornie oddalić się od celu!). Osoby z uszkodzeniami płatów czołowych mają z tym ogromną trudność.

#### 3. Neurobiologia pamięci: dowód na istnienie pamięci proceduralnej (1980)
* W przełomowym eksperymencie **Neal Cohen i Larry Squire (1980)** (a wcześniej badacze pracujący ze słynnym **pacjentem H.M.**, któremu usunięto hipokamp) dawali pacjentom z głęboką amnezją do rozwiązania Wieże Hanoi przez kolejne dni.
* **Wynik zrewolucjonizował naukę o pamięci:** pacjenci każdego dnia twierdzili, że **nigdy wcześniej nie widzieli tej układanki** (całkowity brak *pamięci deklaratywnej / epizodycznej*), a mimo to z dnia na dzień rozwiązywali ją coraz szybciej i w minimalnej liczbie ruchów! Udowodniło to, że mózg posiada osobny system **pamięci proceduralnej** (umiejętności i nawyków, oparty na zwojach podstawy i móżdżku), niezależny od hipokampa.

## 2. Sformułowanie Wież Hanoi jako problemu przestrzeni stanów

Przełóżmy teraz Wieże Hanoi (uogólnione na **$n$ krążków** i **$k$ słupków**) na nasz formalny język przestrzeni stanów:

### Krok 1. Jak zapisać stan? (Sprytna abstrakcja!)
Na pierwszy rzut oka mogłoby się wydawać, że musimy pamiętać zawartość każdego słupka i kolejność krążków na nim.
Ale przypomnijmy sobie regułę nr 2: **większy krążek nigdy nie może leżeć na mniejszym!**

Co z tego wynika? Jeśli wiemy tylko tyle, że na słupku **A** znajdują się krążki nr `1` i `3`, to istnieje **tylko jeden dozwolony sposób**, w jaki mogą na nim leżeć: większy (`3`) musi być na dole, a mniejszy (`1`) na górze!

Dlatego każdy stan wystarczy zapisać jako **krotkę długości $n$**:
$$s = (p_1, p_2, \dots, p_n)$$
gdzie $p_i \in \{\text{A}, \text{B}, \dots\}$ mówi, **na którym słupku leży krążek nr $i$** ($p_1$ = najmniejszy krążek, $p_n$ = największy krążek).
* Np. dla $n=3$ krążków zapis **`(C, B, A)`** oznacza: krążek 1 leży na **C**, krążek 2 na **B**, a największy krążek 3 na **A**.

### Krok 2. Ile jest wszystkich stanów?
Skoro każdy z $n$ krążków może leżeć na dowolnym z $k$ słupków (a ich kolejność na każdym słupku ustala się sama), liczbę wszystkich możliwych stanów dostajemy przez mnożenie:
$$|S| = \underbrace{k \cdot k \cdot \dots \cdot k}_{n \text{ razy}} = k^n$$

* Dla klasycznych 3 słupków i 3 krążków: $|S| = 3^3 = 27$ stanów.
* Dla 4 słupków i 3 krążków: $|S| = 4^3 = 64$ stany.
* A dla **2 słupków** i 3 krążków: $|S| = 2^3 = 8$ stanów. *(Uwaga: za chwilę zobaczysz na grafie, dlaczego przy 2 słupków większość z tych stanów jest **nieosiągalna**!)*

### Krok 3. Stan początkowy i stan docelowy
* **Start:** wszystkie krążki na pierwszym słupku **A**: $(A, A, \dots, A)$.
* **Cel:** wszystkie krążki na ostatnim słupku (np. $(C, C, \dots, C)$ dla 3 słupków).

### Krok 4. Akcje i model przejścia
Każda akcja ma postać **`X → Y`** (*przełóż górny krążek ze słupka X na słupek Y*).
* **Warunek:** na słupku `X` musi leżeć przynajmniej jeden krążek, a najmniejszy krążek na `X` musi mieć mniejszy numer niż najmniejszy krążek na `Y` (lub słupek `Y` jest pusty).
* **Koszt akcji:** `1` za każdy ruch.

---

### Zobaczmy wygenerowaną przestrzeń stanów i graf (Trójkąt Sierpińskiego!)
Uruchom poniższą komórkę dla klasycznych 3 krążków i 3 słupków. Zwróć uwagę na kształt grafu: trzy duże podtrójkąty odpowiadają położeniu największego krążka #3 (na słupku A, B lub C), a przejście między nimi jest możliwe tylko przez pojedyncze „mosty” w narożnikach!

In [ ]:
lh.show_hanoi_space(n_disks=3, n_pegs=3)

### 🔍 Interaktywny eksplorator przestrzeni stanów Wież Hanoi (BFS vs DFS)

W poniższym eksploratorze możesz zmieniać liczbę krążków ($n$) oraz słupków ($k$) i sprawdzić:
1. **Zakładka 📐 Porównanie: BFS vs DFS na grafie** – jak **BFS** idzie prosto wzdłuż krawędzi trójkąta (najkrótsza ścieżka = $7$ kroków dla $n=3, k=3$), podczas gdy **DFS** kluczy zygzakiem po wnętrzu trójkąta ($14$ kroków!).
2. **Zakładka 🎬 Animacja krok po kroku** – podgląd **kolejki FIFO** (w BFS) oraz **stosu LIFO / stosu wywołań** (w DFS) podczas przechodzenia grafu Hanoi.
3. **Co dzieje się dla $k=2$ słupków?** Przesuń suwak *Liczba słupków (k)* na `2` (przy zaznaczonej opcji *Pokaż też stany nieosiągalne*). Zobaczysz, dlaczego bez trzeciego słupka pomocniczego nie da się przełożyć wieży!

In [ ]:
lh.hanoi_space_app()

## 3. 🗼 Ćwiczenie: Wieże Hanoi – przestrzeń stanów, podcele i najkrótsza ścieżka

To ćwiczenie łączy w jedną całość wszystko, czego nauczyliśmy się w **Lab 2** (formalizacja przestrzeni stanów), **Lab 3** (przeszukiwanie grafu BFS i DFS) oraz **Lab 4** (struktura Wież Hanoi, trójkąt Sierpińskiego i analiza środków i celów Newella i Simona).

Program losuje parę układów krążków: **🟢 Stan początkowy (START)** oraz **🟡 Stan docelowy (CEL)** (leżące w różnych podtrójkątach trójkąta Sierpińskiego). Twoje trzy zadania:
1. **Formalizacja stanu początkowego (Lab 2):**
   * **1a.** Zapisz stan **START** jako krotkę $(p_1, \dots, p_n)$, pamiętając, że $p_1$ oznacza słupek **najmniejszego krążka #1**, a $p_n$ – **największego krążka #$n$**.
   * **1b.** Zaznacz wszystkie akcje `X → Y`, które są **dozwolone (stosowalne)** w stanie **START**.
2. **Analiza środków i celów – podcel / most w grafie (Newell i Simon):**
   * Aby przenieść największy krążek **#$n$** ze słupka początkowego na docelowy, trzeba najpierw odsłonić go i zwolnić słupek docelowy – czyli zebrać wszystkie mniejsze krążki ($1 \dots n-1$) na **słupku pomocniczym (buforze)**.
   * Wskaż ten **stan-podcel** (wejście na „most” łączący podtrójkąty w grafie Sierpińskiego) tuż *przed* przełożeniem największego krążka **#$n$**.
3. **Najkrótsza ścieżka `START → CEL` (Lab 3 & 4):**
   * Przekładaj krążki na interaktywnych słupkach po lewej, aby doprowadzić wieżę ze stanu **START** do stanu **CEL** w **minimalnej możliwej liczbie ruchów** (dokładnie takiej, jaką znajduje algorytm **BFS**).

**Dwa tryby** (przełącznik w prawym górnym rogu):
* **📝 Egzamin** – odpowiedzi **🔒 zatwierdzasz raz**; dopiero po zatwierdzeniu zobaczysz swój wynik (`0–3 pkt`), poprawne odpowiedzi oraz tabelę porównującą Twoją ścieżkę z **BFS** i **DFS**. Kolejna próba = **🎲 Nowe zadanie**.
* **🔓 Plac zabaw** – możesz sprawdzać odpowiedzi (**✔ Sprawdź**) wielokrotnie, podejrzeć rozwiązanie (**👁 Pokaż rozwiązanie**), wybierać stan-podcel klikając węzły bezpośrednio na trójkącie Sierpińskiego, przełączyć układ na **klasyczny (`A → C`)** oraz w każdej chwili otworzyć **📊 Porównanie z BFS i DFS** i odtworzyć animacje.

In [ ]:
lh.hanoi_exercise(n_disks=3)   # n_disks: 2–4; seed=2026 → to samo zadanie dla całej grupy; mode="playground" → plac zabaw

### ✏️ Pytania i zadania do ćwiczenia

1. **Rozgrzewka w trybie 📝 Egzamin ($n=2$, potem $n=3$ krążki):**
   * Rozwiąż bezbłędnie (`3 / 3`) dwa losowe zadania dla `2 krążków`, a następnie trzy losowe zadania dla `3 krążków` (przycisk *🎲 Nowe zadanie*).
   * Od czego zależy liczba dozwolonych akcji w stanie **START** (Zadanie 1b)? Kiedy w stanie początkowym dozwolone są tylko **2 akcje**, a kiedy **3 akcje**?
2. **Mosty w trójkącie Sierpińskiego a podcele (Zadanie 2):**
   * Dlaczego w Zadaniu 2 dla $k=3$ słupków istnieje **dokładnie jeden** stan-podcel pozwalający przełożyć największy krążek **#$n$** ze słupka początkowego na docelowy?
   * Po zatwierdzeniu odpowiedzi spójrz na podświetlony na pomarańczowo „most” na grafie Sierpińskiego. Dlaczego każda najkrótsza ścieżka między podtrójkątami musi przejść przez tę jedną krawędź?
3. **Porównanie Twojej ścieżki z BFS i DFS (Zadanie 3):**
   * Po rozwiązaniu zadania dla `3 krążków` kliknij w tabeli porównawczej **▶ Pokaż BFS** oraz **▶ Pokaż DFS** i przełącz widok na mapie grafu (*Twoja / BFS / DFS / Wszystkie*).
   * Dlaczego **BFS** zawsze trafia prosto w most (stan-podcel z Zadania 2), podczas gdy **DFS** kluczy zygzakiem po wnętrzu podtrójkątów i wykonuje często $2\times$ lub $3\times$ więcej ruchów?
4. **Układ klasyczny i $n=4$ krążki (tryb 🔓 Plac zabaw):**
   * Przełącz na **🔓 Plac zabaw**, wybierz `Układ: 🗼 Klasyczny (A → C)` i sprawdź, jak wyglądają odpowiedzi do Zadań 1–3 dla klasycznego problemu $(A, A, A) \to (C, C, C)$.
   * Dla wytrwałych: zmień liczbę krążków na `4 krążki` (81 stanów w grafie!) i rozwiąż jedno losowe zadanie używając metody podcelów.

## 4. Dla ciekawskich: zestawienie BFS i DFS dla różnych $(n, k)$ w kodzie

Poniższa komórka porównuje automatycznie rozmiar przestrzeni stanów ($k^n$), długość najkrótszej ścieżki (BFS) oraz długość ścieżki DFS dla różnych kombinacji liczby krążków i słupków:

In [ ]:
print(f"{'Krążki (n)':>10} | {'Słupki (k)':>10} | {'Stany (k^n)':>11} | {'BFS: ruchy (odw.)':>20} | {'DFS: ruchy (odw.)':>20}")
print("-" * 83)
for k in (2, 3, 4):
    for n in (2, 3, 4):
        p = lh.Hanoi(n_disks=n, n_pegs=k)
        bfs_end = lsearch.trace_bfs_iterative(p)[-1]
        dfs_end = lsearch.trace_dfs_iterative(p)[-1]
        b_len = len(bfs_end.solution) if bfs_end.solution is not None else "brak"
        d_len = len(dfs_end.solution) if dfs_end.solution is not None else "brak"
        print(f"{n:>10} | {k:>10} | {k**n:>11} | {str(b_len):>6} ({len(bfs_end.visited):>3} st.)    | {str(d_len):>6} ({len(dfs_end.visited):>3} st.)")